# Segmento 2 — Cosa possiamo fare senza un LLM?

Abbiamo visto il dataset. Ora vediamo fin dove possiamo arrivare con Python puro — niente IA, niente machine learning, solo codice.

In [1]:
from datasets import load_dataset
import ast, re
import pandas as pd
from collections import Counter

ds = load_dataset("Hieu-Pham/kaggle_food_recipes", split="train")
df = ds.to_pandas().drop(columns=["Unnamed: 0", "Image_Name"])

# Parse all ingredient lists once
df["parsed_ingredients"] = df["Ingredients"].apply(ast.literal_eval)
print(f"{len(df)} recipes loaded")

/home/federios/aidea/boolean-master/boolean-master-demo/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


13501 recipes loaded


## Quali sono gli ingredienti più comuni?

Uniamo tutte le liste di ingredienti in un unico grande insieme e contiamoli. Secondo voi qual è il numero 1?

In [2]:
# Flatten all ingredient strings into one list and count
all_ingredients = [ing.lower().strip() for ings in df["parsed_ingredients"] for ing in ings]
ingredient_counts = Counter(all_ingredients)

print(f"{len(all_ingredients):,} total ingredient strings across all recipes")
print(f"{len(ingredient_counts):,} unique strings\n")
print("Top 20 most common:")
for i, (ing, count) in enumerate(ingredient_counts.most_common(20), 1):
    print(f"  {i:2d}. {ing} ({count:,} recipes)")

143,693 total ingredient strings across all recipes
73,078 unique strings

Top 20 most common:
   1. kosher salt (1,138 recipes)
   2. 1/2 teaspoon salt (666 recipes)
   3. kosher salt, freshly ground pepper (659 recipes)
   4. freshly ground black pepper (657 recipes)
   5. 1/4 teaspoon salt (612 recipes)
   6. 2 tablespoons olive oil (497 recipes)
   7. 2 large eggs (440 recipes)
   8. 1 teaspoon vanilla extract (405 recipes)
   9. 1 teaspoon salt (385 recipes)
  10. 1 large egg (362 recipes)
  11. 1 tablespoon olive oil (339 recipes)
  12. 1/2 cup sugar (338 recipes)
  13. 1 cup sugar (316 recipes)
  14. 2 tablespoons fresh lemon juice (304 recipes)
  15. 1 tablespoon fresh lemon juice (300 recipes)
  16. 1 teaspoon kosher salt (286 recipes)
  17. nonstick vegetable oil spray (285 recipes)
  18. 1/2 teaspoon kosher salt (285 recipes)
  19. 1/4 cup olive oil (268 recipes)
  20. 1/4 cup sugar (266 recipes)


## Ricerca per ingredienti — "Cosa posso preparare?"

Un caso d'uso classico: avete alcuni ingredienti a casa, quali ricette corrispondono? Una semplice ricerca per sottostringhe ci porta sorprendentemente lontano.

In [3]:
def search_by_ingredients(keywords, df=df, top_n=10):
    """Find recipes whose ingredient list contains ALL the given keywords."""
    keywords = [k.lower() for k in keywords]
    
    def matches(parsed_ings):
        text = " ".join(parsed_ings).lower()
        return all(k in text for k in keywords)
    
    hits = df[df["parsed_ingredients"].apply(matches)]
    print(f"Found {len(hits)} recipes with {keywords}\n")
    for _, row in hits.head(top_n).iterrows():
        n = len(row["parsed_ingredients"])
        print(f"  - {row['Title']} ({n} ingredients)")

search_by_ingredients(["chicken", "lemon", "garlic"])

Found 267 recipes with ['chicken', 'lemon', 'garlic']

  - Caesar Salad Roast Chicken (13 ingredients)
  - Chicken and Rice With Leeks and Salsa Verde (13 ingredients)
  - Chicken and Potato Gratin With Brown Butter Cream (13 ingredients)
  - Stuffed Eggplants and Zucchini in a Rich Tomato Sauce (Baatingan w Kusaa Bil Banadoura) (31 ingredients)
  - Chicken Meatballs With Molokhieh, Garlic, and Cilantro (25 ingredients)
  - Crispy Turmeric-and-Pepper-Spiced Chicken Wings (10 ingredients)
  - Green-Garlic-Rubbed Buttery Roast Chicken (6 ingredients)
  - Chicken Zucchini Burgers (24 ingredients)
  - Chicken Spiedies (Marinated Chicken on a Bun) (16 ingredients)
  - Instant Pot Lemon Chicken With Garlic and Olives (9 ingredients)


## Che tipo di cottura c'è in questo dataset?

Possiamo contare i verbi di cottura più comuni nelle istruzioni per vedere quali tecniche predominano.

In [4]:
cooking_verbs = [
    "bake", "roast", "grill", "fry", "sauté", "simmer", "boil", "steam",
    "broil", "braise", "poach", "stir", "whisk", "chop", "dice", "slice",
    "mince", "blend", "fold", "knead", "marinate", "season", "garnish",
]

# Count how many recipes mention each verb in their instructions
all_instructions = df["Instructions"].dropna().str.lower()
verb_counts = {
    verb: all_instructions.str.contains(verb, regex=False).sum()
    for verb in cooking_verbs
}

print("Cooking techniques by frequency:\n")
for verb, count in sorted(verb_counts.items(), key=lambda x: -x[1]):
    bar = "█" * (count // 200)
    print(f"  {verb:<10s} {count:>5,} recipes  {bar}")

Cooking techniques by frequency:

  stir       7,757 recipes  ██████████████████████████████████████
  season     5,164 recipes  █████████████████████████
  boil       4,558 recipes  ██████████████████████
  whisk      4,307 recipes  █████████████████████
  bake       3,780 recipes  ██████████████████
  simmer     3,612 recipes  ██████████████████
  blend      2,943 recipes  ██████████████
  slice      2,887 recipes  ██████████████
  chop       2,116 recipes  ██████████
  roast      1,637 recipes  ████████
  fold       1,521 recipes  ███████
  sauté      1,378 recipes  ██████
  garnish    1,215 recipes  ██████
  grill      1,109 recipes  █████
  fry          795 recipes  ███
  broil        517 recipes  ██
  steam        468 recipes  ██
  knead        385 recipes  █
  marinate     363 recipes  █
  mince        225 recipes  █
  dice         192 recipes  
  braise       127 recipes  
  poach        121 recipes  


## Possiamo fare il parsing degli ingredienti con le regex?

Proviamo a estrarre quantità, unità di misura e nome dell'ingrediente da ogni stringa. Scriviamo una semplice regex e vediamo fin dove arriviamo.

In [6]:
# Matches: <number> <unit> <everything else>
# e.g. "2 cups all-purpose flour" → qty="2", unit="cups", ingredient="all-purpose flour"
UNITS = r"(cups?|teaspoons?|tablespoons?|tsp|tbsp|pounds?|lb|oz|ounces?|cloves?|pinch)"

def parse_ingredient(text):
    """Try to extract (quantity, unit, ingredient) with regex."""
    pattern = rf"^([\d½¼¾⅓⅔/.\s]+)\s+{UNITS}\.?\s+(.+)"
    m = re.match(pattern, text.strip(), re.IGNORECASE)
    if m:
        return {"quantity": m.group(1).strip(), "unit": m.group(2).strip(), "ingredient": m.group(3).strip()}
    return None

# Examples where regex works well
easy_examples = [
    "2 cups all-purpose flour",
    "1 teaspoon vanilla extract",
    "3 tablespoons olive oil",
    "4 cloves garlic",
    "1/2 cup sugar",
    "2 pounds boneless chicken breast",
]

print("Regex works great on clean inputs:\n")
for ing in easy_examples:
    result = parse_ingredient(ing)
    print(f"  '{ing}'")
    print(f"    → {result}\n")

Regex works great on clean inputs:

  '2 cups all-purpose flour'
    → {'quantity': '2', 'unit': 'cups', 'ingredient': 'all-purpose flour'}

  '1 teaspoon vanilla extract'
    → {'quantity': '1', 'unit': 'teaspoon', 'ingredient': 'vanilla extract'}

  '3 tablespoons olive oil'
    → {'quantity': '3', 'unit': 'tablespoons', 'ingredient': 'olive oil'}

  '4 cloves garlic'
    → {'quantity': '4', 'unit': 'cloves', 'ingredient': 'garlic'}

  '1/2 cup sugar'
    → {'quantity': '1/2', 'unit': 'cup', 'ingredient': 'sugar'}

  '2 pounds boneless chicken breast'
    → {'quantity': '2', 'unit': 'pounds', 'ingredient': 'boneless chicken breast'}



## Ora proviamo con dati reali

Queste sono stringhe di ingredienti reali dal dataset. Stessa regex. Vediamo cosa succede.

In [7]:
# Real ingredient strings from the dataset that break the regex
hard_examples = [
    "Pinch of crushed red pepper flakes",
    "1 (3½–4-lb.) whole chicken",
    "One 14-ounce can whole peeled tomatoes",
    "Salt and pepper",
    "Vegetable oil, for frying",
    "Juice of 2 lemons",
    "6 Tbsp. unsalted butter, melted, plus 3 Tbsp. room temperature",
]

# What a human would extract (same fields: quantity, unit, ingredient)
human_answers = [
    {"quantity": "1",    "unit": "pinch",       "ingredient": "crushed red pepper flakes"},
    {"quantity": "1",    "unit": "—",           "ingredient": "(3½–4-lb.) whole chicken"},
    {"quantity": "1",    "unit": "14-ounce can", "ingredient": "whole peeled tomatoes"},
    {"quantity": "—",    "unit": "—",           "ingredient": "salt and pepper"},
    {"quantity": "—",    "unit": "—",           "ingredient": "vegetable oil"},
    {"quantity": "2",    "unit": "—",           "ingredient": "lemon juice"},
    {"quantity": "6+3",  "unit": "Tbsp",        "ingredient": "unsalted butter"},
]

print("Real ingredient strings vs. our regex:\n")
for ing, human in zip(hard_examples, human_answers):
    result = parse_ingredient(ing)
    print(f"  '{ing}'")
    if result:
        print(f"    Regex: qty={result['quantity']}, unit={result['unit']}, ing={result['ingredient']}")
    else:
        print(f"    Regex: (no match)")
    print(f"    Human: qty={human['quantity']}, unit={human['unit']}, ing={human['ingredient']}")
    print()

Real ingredient strings vs. our regex:

  'Pinch of crushed red pepper flakes'
    Regex: (no match)
    Human: qty=1, unit=pinch, ing=crushed red pepper flakes

  '1 (3½–4-lb.) whole chicken'
    Regex: (no match)
    Human: qty=1, unit=—, ing=(3½–4-lb.) whole chicken

  'One 14-ounce can whole peeled tomatoes'
    Regex: (no match)
    Human: qty=1, unit=14-ounce can, ing=whole peeled tomatoes

  'Salt and pepper'
    Regex: (no match)
    Human: qty=—, unit=—, ing=salt and pepper

  'Vegetable oil, for frying'
    Regex: (no match)
    Human: qty=—, unit=—, ing=vegetable oil

  'Juice of 2 lemons'
    Regex: (no match)
    Human: qty=2, unit=—, ing=lemon juice

  '6 Tbsp. unsalted butter, melted, plus 3 Tbsp. room temperature'
    Regex: qty=6, unit=Tbsp, ing=unsalted butter, melted, plus 3 Tbsp. room temperature
    Human: qty=6+3, unit=Tbsp, ing=unsalted butter

